# ============================================================
# Spotify Song Charts Big Data Processing Pipeline
# Bronze → Silver → Gold
# ============================================================

In [0]:
from pyspark.sql.functions import *

from pyspark.sql.window import Window

from pyspark.sql.types import *

In [0]:
base_path = "/Volumes/project/datasets/spotify/"
albums_df = spark.read.csv(
    base_path+"albums.csv",
    header=True,
    inferSchema=True
)

artists_df = spark.read.csv(
    base_path+"artists.csv",
    header=True,
    inferSchema=True
)

songs_df = spark.read.csv(
    base_path+"songs.csv",
    header=True,
    inferSchema=True
)

In [0]:
bronze_song_charts = spark.read.parquet(
    "/Volumes/project/datasets/spotify/charts_songs_daily.parquet"
)

In [0]:
print("Rows:", bronze_song_charts.count())

print("Columns:", len(bronze_song_charts.columns))

Rows: 42757018
Columns: 18


In [0]:
bronze_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)



In [0]:
display(
    bronze_song_charts.limit(10)
)

date,country,rank,uri,artist_names,track_name,label,peak_rank,previous_rank,days_on_chart,streams,consecutive_days,entry_status,peak_date,entry_rank,entry_date,release_date,artist_uris
2019-06-04,ae,1,spotify:track:6u7jPi22kF8CTQ3rb9DHE7,Lil Nas X|Billy Ray Cyrus,Old Town Road (feat. Billy Ray Cyrus) - Remix,Columbia,1,-1,1,4230,1,NEW_ENTRY,2019-06-04,1,2019-06-04,2019-06-21,spotify:artist:7jVv8c5Fj3E9VhNjxT4snq|spotify:artist:60rpJ9SgigSd16DOAG7GSa
2019-06-04,ae,2,spotify:track:3HVWdVOQ0ZA45FuZGSfvns,Ed Sheeran & Justin Bieber|Justin Bieber,I Don't Care (with Justin Bieber),Atlantic Records UK,2,-1,1,3806,1,NEW_ENTRY,2019-06-04,2,2019-06-04,2019-05-10,spotify:artist:1WsYCXdezMjn0KoIrLvMmC|spotify:artist:1uNFoZAHBGtllmzznpCI3s
2019-06-04,ae,3,spotify:track:2Fxmhks0bxGSBdJ92vM42m,Billie Eilish,bad guy,Darkroom/Interscope Records,3,-1,1,3527,1,NEW_ENTRY,2019-06-04,3,2019-06-04,2019-03-29,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH
2019-06-04,ae,4,spotify:track:3KkXRkHbMCARz0aVfEt68P,Post Malone|Swae Lee,Sunflower - Spider-Man: Into the Spider-Verse,Universal Records,4,-1,1,2955,1,NEW_ENTRY,2019-06-04,4,2019-06-04,2018-12-14,spotify:artist:246dkjvS1zLTtiykXe5h60|spotify:artist:1zNqQNIdeOUZHb8zbZRFMX
2019-06-04,ae,5,spotify:track:53CJANUxooaqGOtdsBTh7O,Lil Nas X,Old Town Road,Columbia,5,-1,1,2619,1,NEW_ENTRY,2019-06-04,5,2019-06-04,2019-06-21,spotify:artist:7jVv8c5Fj3E9VhNjxT4snq
2019-06-04,ae,6,spotify:track:6LsAAHotRLMOHfCsSfYCsz,Shawn Mendes,If I Can't Have You,Island Records,6,-1,1,2536,1,NEW_ENTRY,2019-06-04,6,2019-06-04,2019-05-03,spotify:artist:7n2wHs1TKAczGzO7Dd2rGr
2019-06-04,ae,7,spotify:track:7DcvwMAiqKJQD1rrdfxSDx,Young Thug|J. Cole|Travis Scott,The London (feat. J. Cole & Travis Scott),300 Entertainment/Atl,7,-1,1,2493,1,NEW_ENTRY,2019-06-04,7,2019-06-04,2019-05-23,spotify:artist:50co4Is1HCEo8bhOyUWKpn|spotify:artist:6l3HvQ5sa6mXTsMTB19rO5|spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY
2019-06-04,ae,8,spotify:track:4y3OI86AEP6PQoDE6olYhO,Jonas Brothers,Sucker,Jonas Brothers Recording,8,-1,1,1999,1,NEW_ENTRY,2019-06-04,8,2019-06-04,2019-06-07,spotify:artist:7gOdHgIoIKoe4i9Tta6qdD
2019-06-04,ae,9,spotify:track:6TqXcAFInzjp0bODyvrWEq,Khalid|Disclosure,Talk (feat. Disclosure),"Right Hand Music Group, LLC/RCA Records",9,-1,1,1882,1,NEW_ENTRY,2019-06-04,9,2019-06-04,2019-04-05,spotify:artist:6LuN9FCkKOj5PcnpouEgny|spotify:artist:6nS5roXSAGhTGr34W6n7Et
2019-06-04,ae,10,spotify:track:2dpaYNEQHiRxtZbfNsse99,Marshmello|Bastille,Happier,Joytime Collective,10,-1,1,1846,1,NEW_ENTRY,2019-06-04,10,2019-06-04,2018-08-17,spotify:artist:64KEffDW9EtZ1y2vBYgq8T|spotify:artist:7EQ0qTo7fWT7DPxmxtSYEc


In [0]:
duplicate_records = (

bronze_song_charts.count()

-

bronze_song_charts
.dropDuplicates()
.count()

)


print(
"Duplicate Records:",
duplicate_records
)

Duplicate Records: 0


In [0]:
display(
    bronze_song_charts.describe()
)

summary,country,rank,uri,artist_names,track_name,label,peak_rank,previous_rank,days_on_chart,streams,consecutive_days,entry_status,entry_rank,artist_uris
count,42757018,42757018,42757018,42712307,42712307,42714025,42757018,42757018,42757018,42757018,42757018,42757018,42757018,42757018
mean,null,98.18277088453642,null,650.6540295119182,Infinity,1.3623392226439463E7,34.44040110561499,87.21173822271703,266.2623929245954,67799.69933520621,129.18163677831788,null,102.53703714791335,null
stddev,null,57.728287623528445,null,507.5860369544343,NaN,1.6346008088266988E9,39.41418283234047,58.59409690234409,358.86676646747395,243949.62129905046,206.78136576272036,null,70.07538717091504,null
min,ae,1,spotify:track:000N4CJL8IjQ0f2I4grgBO,jambino|Loco|HyunA,!,"""Hitmen Music / Top Notch Music BV",1,-1,1,1001,1,MOVED_DOWN,1,spotify:artist:000BblCiHJeKvtiq5aiHOs
max,za,200,spotify:track:7zzhknA0A39TH81meuX7WA,하숙자,🪐,피네이션,200,200,3427,30987370,3427,RE_ENTRY,200,spotify:artist:7zzosAlsXxJJ5vXPymZZAj|spotify:artist:7DngHhdutSXfKWLa34MngI


In [0]:
business_key_check = (

bronze_song_charts

.select(

count("*")
.alias("total_records"),

countDistinct(
    "date",
    "country",
    "uri"
)
.alias("unique_song_events")

)

)


display(
business_key_check
)

total_records,unique_song_events
42757018,42756826


Distinct Value Profiling

In [0]:
bronze_song_charts.select(

countDistinct("country")
.alias("total_countries")

).show()

+---------------+
|total_countries|
+---------------+
|             73|
+---------------+



In [0]:
bronze_song_charts.select(

countDistinct("uri")
.alias("unique_songs")

).show()

+------------+
|unique_songs|
+------------+
|      241874|
+------------+



In [0]:
bronze_song_charts.select(

countDistinct("artist_names")
.alias("unique_artists")

).show()

+--------------+
|unique_artists|
+--------------+
|         93072|
+--------------+



In [0]:
bronze_summary = [

(
"Records",
bronze_song_charts.count()
),

(
"Columns",
len(bronze_song_charts.columns)
),

(
"Countries",
bronze_song_charts
.select(countDistinct("country"))
.collect()[0][0]
),

(
"Songs",
bronze_song_charts
.select(countDistinct("uri"))
.collect()[0][0]
)

]


display(

spark.createDataFrame(
bronze_summary,
["Metric","Value"]
)

)

Metric,Value
Records,42757018
Columns,18
Countries,73
Songs,241874


In [0]:
missing_song_metadata = (
    bronze_song_charts
    .select(
        col("uri").alias("track_uri")
    )
    .distinct()
    .join(
        songs_df.select("track_uri"),
        on="track_uri",
        how="left_anti"
    )
)


missing_song_metadata.count()

34710

In [0]:
bronze_song_charts.select(
    countDistinct("artist_names")
    .alias("unique_chart_artists")
).show()

+--------------------+
|unique_chart_artists|
+--------------------+
|               93072|
+--------------------+



In [0]:
uncharted_songs = (

    songs_df
    .select("track_uri")
    .distinct()

    .join(
        bronze_song_charts
        .select(
            col("uri")
            .alias("track_uri")
        )
        .distinct(),

        on="track_uri",
        how="left_anti"
    )

)


uncharted_songs.count()

0

Artists Profiling

In [0]:
chart_artists = (
    bronze_song_charts
    .select(
        explode(
            split("artist_uris", ",")
        ).alias("artist_uri")
    )
    .withColumn(
        "artist_uri",
        trim(col("artist_uri"))
    )
    .distinct()
)


chart_artists.count()

93457

In [0]:
artists_df.select(
    countDistinct("artist_uri")
    .alias("metadata_artists")
).show()

+----------------+
|metadata_artists|
+----------------+
|           63775|
+----------------+



In [0]:
missing_artist_metadata = (

chart_artists

.join(
    artists_df
    .select("artist_uri")
    .distinct(),

    on="artist_uri",
    how="left_anti"
)

)


missing_artist_metadata.count()

66541

In [0]:
artists_not_charted = (

artists_df
.select("artist_uri")
.distinct()

.join(

chart_artists,

on="artist_uri",
how="left_anti"

)

)


artists_not_charted.count()

36859

In [0]:
artist_summary = [

(
"Artists Metadata",
artists_df.select(
countDistinct("artist_uri")
).collect()[0][0]
),

(
"Chart Artists",
chart_artists.count()
),

(
"Chart Artists Missing Metadata",
missing_artist_metadata.count()
),

(
"Metadata Artists Not Charted",
artists_not_charted.count()
)

]


display(
spark.createDataFrame(
artist_summary,
["Metric","Value"]
)
)

Metric,Value
Artists Metadata,63775
Chart Artists,93457
Chart Artists Missing Metadata,66541
Metadata Artists Not Charted,36859


Labels

In [0]:
chart_labels = (
    bronze_song_charts
    .select(trim("label").alias("label"))
    .filter(col("label").isNotNull())
    .distinct()
)

metadata_labels = (
    songs_df
    .select(trim("label").alias("label"))
    .filter(col("label").isNotNull())
    .distinct()
)


label_summary = [
    ("Metadata Labels", metadata_labels.count()),
    ("Chart Labels", chart_labels.count()),
    ("Chart Labels Missing Metadata", chart_labels.join(metadata_labels,"label","left_anti").count()),
    ("Metadata Labels Not Charted", metadata_labels.join(chart_labels,"label","left_anti").count())
]


display(
    spark.createDataFrame(label_summary, ["Metric","Value"])
)

Metric,Value
Metadata Labels,27580
Chart Labels,27947
Chart Labels Missing Metadata,405
Metadata Labels Not Charted,38


BRONZE LAYER DATASET SUMMARY
Dataset: Spotify Daily Song Charts
Source Table: charts_songs_daily


Metric                  Actual Value        Source                          Meaning
------------------------------------------------------------------------------------------------------------

Total Chart Records      42,757,018         charts_songs_daily              Daily song performance events

Unique Songs             241,874            charts_songs_daily.uri          Songs that appeared in Spotify charts

Unique Artists           93,457             charts_songs_daily.artist_uris  Artists that appeared in song charts

Unique Labels            27,947             charts_songs_daily.label        Music labels that appeared in charts

Countries Covered        73                 charts_songs_daily.country      Spotify markets/countries covered


Additional Dataset Information
------------------------------------------------------------------------------------------------------------

Time Coverage            2017-01-01 to 2026-05-20

Total Columns            18

Data Granularity         One record represents one song's performance
                         in one country on one particular date


Business Key
------------------------------------------------------------------------------------------------------------

Unique Event Identifier:

date + country + uri


Example:

2026-05-20 + US + spotify_track_uri
=
One song chart performance record


Bronze Layer Findings
------------------------------------------------------------------------------------------------------------

1. No exact duplicate records found.

2. 192 duplicate business events identified using:
   date + country + uri

3. Minor missing values detected in:
   - track_name
   - artist_names
   - label
   - release_date

4. Entity relationship validation performed for:
   - Songs
   - Artists
   - Labels

5. Complete chart history will be preserved.
   Data quality issues will be handled in Silver Layer.